# 🔁 Module 4 — Recurrent Networks & Sequence Modelling
### Introduction to Deep Learning | AgenticLabs.ng

---

## 🎯 Learning Objectives
- Understand why sequences need a different architecture
- Implement a **Vanilla RNN** from scratch to build intuition
- Understand the **vanishing gradient problem** and why it limits RNNs
- Learn how **LSTMs** and **GRUs** solve this problem
- Build a **sentiment analysis** model on the IMDb movie review dataset
- Understand **word embeddings** and how they represent meaning

---

## 4.1 — Why Do Sequences Need Special Treatment?

MLPs and CNNs treat every input independently. But language, time series, and audio are **sequential**: the meaning of a word depends on the words before it.

> *"The trophy didn't fit in the bag because it was too **big**."*

The word "big" refers to "trophy", not "bag" — to understand this, the model must remember context from earlier in the sentence. This is what recurrent networks are designed for.


## 4.2 — The Vanilla RNN: Core Idea

At each step, an RNN reads the current input AND passes a hidden state (memory) from the previous step.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ── Manual RNN to understand the hidden state ──────────────
class VanillaRNN:
    """
    At each timestep t:
      h_t = tanh(W_hh * h_{t-1} + W_xh * x_t + b_h)
      y_t = W_hy * h_t + b_y
    """
    def __init__(self, input_size, hidden_size, output_size):
        scale = 0.01
        self.W_xh = np.random.randn(input_size, hidden_size) * scale
        self.W_hh = np.random.randn(hidden_size, hidden_size) * scale
        self.W_hy = np.random.randn(hidden_size, output_size) * scale
        self.b_h  = np.zeros(hidden_size)
        self.b_y  = np.zeros(output_size)
    
    def forward(self, inputs):
        """inputs: list of vectors (one per timestep)"""
        h = np.zeros(self.W_hh.shape[0])
        hidden_states, outputs = [], []
        
        for x in inputs:
            h = np.tanh(x @ self.W_xh + h @ self.W_hh + self.b_h)
            y = h @ self.W_hy + self.b_y
            hidden_states.append(h.copy())
            outputs.append(y)
        
        return outputs, hidden_states

# ── Demonstrate: processing a short sequence ─────────────
rnn = VanillaRNN(input_size=5, hidden_size=10, output_size=3)
# Simulate 8 timesteps (e.g. 8 words), each encoded as a 5-dim vector
sequence = [np.random.randn(5) for _ in range(8)]
outputs, hidden_states = rnn.forward(sequence)

print("Sequence length:", len(sequence))
print("Output at each step shape:", outputs[0].shape)
print("Hidden state shape:", hidden_states[0].shape)

# Visualise hidden state evolution
hs_matrix = np.array(hidden_states)
plt.figure(figsize=(10, 3))
plt.imshow(hs_matrix.T, cmap='RdBu', aspect='auto')
plt.colorbar(label='Activation')
plt.xlabel("Timestep"); plt.ylabel("Hidden unit")
plt.title("Hidden State Evolution Across Timesteps")
plt.tight_layout(); plt.show()


## 4.3 — The Vanishing Gradient Problem

During backpropagation through time (BPTT), gradients must pass through every timestep. If they are repeatedly multiplied by small values (< 1), they **vanish** exponentially. This means RNNs struggle to learn long-range dependencies.

In [ ]:
# ── Vanishing gradient demonstration ─────────────────────
def simulate_gradient_flow(n_steps, weight):
    """Simulate gradient magnitude as it flows back through n_steps"""
    grad = 1.0
    grad_history = [grad]
    for _ in range(n_steps):
        grad *= weight      # each step multiplies by the weight
        grad_history.append(abs(grad))
    return grad_history

steps = 50
plt.figure(figsize=(10, 4))
for w, label, color in [(0.5, 'w=0.5 (vanish)', 'red'),
                          (0.9, 'w=0.9 (slow vanish)', 'orange'),
                          (1.0, 'w=1.0 (stable)', 'green'),
                          (1.1, 'w=1.1 (explode)', 'blue')]:
    hist = simulate_gradient_flow(steps, w)
    plt.plot(hist, label=label, color=color, linewidth=2)

plt.yscale('log')
plt.xlabel("Timestep (going backwards)"); plt.ylabel("Gradient magnitude (log)")
plt.title("Vanishing vs Exploding Gradients in RNNs")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.show()

print("This is WHY we need LSTMs — they use gating to control gradient flow.")


## 4.4 — LSTM: Long Short-Term Memory

The LSTM (Hochreiter & Schmidhuber, 1997) solves the vanishing gradient problem with **gates**:

| Gate | Function |
|---|---|
| **Forget gate** | What to erase from memory |
| **Input gate** | What new information to store |
| **Output gate** | What to output from memory |
| **Cell state** | Long-term memory that passes through largely unchanged |

The cell state is like a conveyor belt — information can flow with very little modification.


In [ ]:
# ── LSTM Sentiment Analysis on IMDb ───────────────────────
# Download IMDb dataset (we use a pre-tokenised version for simplicity)
!pip install datasets --quiet
from datasets import load_dataset

dataset = load_dataset("imdb")
print(f"Train: {len(dataset['train'])} | Test: {len(dataset['test'])}")
print("\nSample review:")
print(dataset['train'][0]['text'][:300], "...")
print("\nSentiment:", "Positive" if dataset['train'][0]['label']==1 else "Negative")


In [ ]:
# ── Tokenise and build vocabulary ────────────────────────
import re
from collections import Counter

def tokenise(text):
    text = re.sub(r'<.*?>', ' ', text.lower())
    return re.sub(r'[^a-z\s]', '', text).split()

# Build vocab from training set
print("Building vocabulary...")
all_words = []
for item in dataset['train']:
    all_words.extend(tokenise(item['text']))

vocab_counter = Counter(all_words)
VOCAB_SIZE = 10000
vocab  = ['<PAD>', '<UNK>'] + [w for w, _ in vocab_counter.most_common(VOCAB_SIZE-2)]
w2idx  = {w: i for i, w in enumerate(vocab)}

print(f"Vocabulary size: {len(vocab):,}")
print(f"Most common words: {[w for w,_ in vocab_counter.most_common(10)]}")


In [ ]:
# ── Dataset & DataLoader ──────────────────────────────────
MAX_LEN = 256

class IMDbDataset(Dataset):
    def __init__(self, split):
        self.data = dataset[split]
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        tokens = tokenise(self.data[idx]['text'])
        ids    = [w2idx.get(t, 1) for t in tokens[:MAX_LEN]]   # 1 = <UNK>
        # Pad or truncate to MAX_LEN
        if len(ids) < MAX_LEN:
            ids += [0] * (MAX_LEN - len(ids))   # 0 = <PAD>
        return torch.tensor(ids, dtype=torch.long), torch.tensor(self.data[idx]['label'])

train_ds = IMDbDataset('train')
test_ds  = IMDbDataset('test')
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=64, shuffle=False)

print(f"Training batches: {len(train_dl)}")
print(f"Test batches    : {len(test_dl)}")


In [ ]:
# ── LSTM Model ───────────────────────────────────────────
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256,
                 n_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, n_layers,
                            batch_first=True, dropout=dropout, bidirectional=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),   # *2 for bidirectional
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )
    
    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        # Concatenate last forward and backward hidden states
        out = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.fc(out).squeeze(1)

sentiment_model = SentimentLSTM(len(vocab)).to(device)
total_params = sum(p.numel() for p in sentiment_model.parameters())
print(sentiment_model)
print(f"\nTotal parameters: {total_params:,}")


In [ ]:
# ── Train the LSTM ────────────────────────────────────────
criterion = nn.BCEWithLogitsLoss()
optimiser = optim.Adam(sentiment_model.parameters(), lr=0.001)

train_losses, test_accs = [], []

for epoch in range(1, 6):
    sentiment_model.train()
    epoch_loss = 0
    for X, y in train_dl:
        X, y = X.to(device), y.float().to(device)
        optimiser.zero_grad()
        out  = sentiment_model(X)
        loss = criterion(out, y)
        loss.backward()
        nn.utils.clip_grad_norm_(sentiment_model.parameters(), 1.0)  # gradient clipping
        optimiser.step()
        epoch_loss += loss.item()
    
    # Evaluate
    sentiment_model.eval()
    correct = total = 0
    with torch.no_grad():
        for X, y in test_dl:
            X, y = X.to(device), y.float().to(device)
            preds = (torch.sigmoid(sentiment_model(X)) > 0.5).long()
            correct += (preds == y.long()).sum().item()
            total   += y.size(0)
    
    avg_loss = epoch_loss / len(train_dl)
    acc = correct / total * 100
    train_losses.append(avg_loss)
    test_accs.append(acc)
    print(f"Epoch {epoch}/5 | Loss: {avg_loss:.4f} | Test Accuracy: {acc:.2f}%")

print(f"\nBest Test Accuracy: {max(test_accs):.2f}%")


In [ ]:
# ── Test on custom reviews ────────────────────────────────
def predict_sentiment(review, model, w2idx, max_len=256):
    model.eval()
    tokens = tokenise(review)
    ids    = [w2idx.get(t, 1) for t in tokens[:max_len]]
    if len(ids) < max_len:
        ids += [0] * (max_len - len(ids))
    x = torch.tensor([ids], dtype=torch.long).to(device)
    with torch.no_grad():
        prob = torch.sigmoid(model(x)).item()
    label = "POSITIVE 😊" if prob > 0.5 else "NEGATIVE 😞"
    return label, prob

reviews = [
    "This film was absolutely brilliant! The acting was superb and the story was captivating from start to finish.",
    "Terrible movie. Boring, predictable plot with wooden acting. Complete waste of two hours.",
    "It was okay, had some good moments but the ending was very disappointing.",
]
print("Sentiment Analysis on Custom Reviews")
print("=" * 55)
for review in reviews:
    label, prob = predict_sentiment(review, sentiment_model, w2idx)
    print(f"\nReview: {review[:70]}...")
    print(f"Result : {label} (confidence: {prob:.3f})")


## 📝 Exercises

1. **GRU vs LSTM**: Replace `nn.LSTM` with `nn.GRU` (fewer parameters). How does accuracy compare?
2. **Sequence length**: Try `MAX_LEN=64` and `MAX_LEN=512`. What is the tradeoff between speed and accuracy?
3. **Pretrained embeddings**: Replace the random embedding layer with GloVe embeddings (available via `torchtext`). Does it help?
4. **Visualise attention weights**: After training, extract the hidden states and visualise which words the model activates most on for a positive and a negative review.

---

## ✅ Module 4 Summary

| Concept | Key takeaway |
|---|---|
| RNN | Processes sequences with a shared hidden state (memory) |
| Vanishing gradient | Gradients shrink through many timesteps — long context is lost |
| LSTM | Gating mechanisms allow selective memory over long sequences |
| Bidirectional | Process sequence both forward and backward for richer context |
| Word embeddings | Dense vector representations that capture word meaning |

**Next up → Module 5: Training Best Practices**
